# (부록) 모델·공급자별 Function Calling 신뢰도 비교 — NVIDIA build vs OpenRouter

> 본 노트북은 `M02_3_mcp_a2a.ipynb` **섹션 7** 의 보충 자료입니다.
> *"도구 호출이 되느냐"* 는 **모델 크기만으로 결정되지 않으며**, **모델 특성**과 **공급자(서비스) 가용성**에도
> 좌우된다는 점을 **반복 실험**으로 **정량 비교**합니다.

## 비교 대상
| # | 공급자 · 모델 | 특징 |
|---|---|---|
| ① | **NVIDIA build** · `deepseek-ai/deepseek-v4.1-flash` ★기본 | 모델 고정, 빠른 응답, 병렬 도구 호출 |
| ② | **OpenRouter** · `liquid/lfm-2.5-2.6b:free` (소형 2.6B) | 작은 모델도 도구 호출을 하는지 확인 |
| ③ | **OpenRouter** · `nvidia/nemotron-3-super-120b-a12b:free` (대형) | 큰 모델이 두 도구를 한 번에 부르는지 확인 |
| ④ | **OpenRouter** · `openrouter/free` (무료 모델 자동 라우팅) | 호출마다 응답 모델이 바뀔 수 있음 |

## 핵심 포인트 — 세 축으로 읽기
- **모델 크기 축** → ② vs ③: *큰 모델일수록 무조건 낫다* 가 아닙니다. 모델마다 도구 호출 학습 방식이 달라서,
  작은 모델이 두 도구를 모두 부르고 큰 모델이 하나만 부르는 경우도 있습니다.
- **공급자 가용성 축** → ②·③ vs ④: 특정 `:free` 모델을 고정하면 과부하(503)·요청 한도(429)로 `ERROR` 가
  섞일 수 있습니다. 라우터(`openrouter/free`)는 사용 가능한 모델로 넘어가 가용성이 높지만, 결과 재현성은 낮아집니다.
- **멀티(병렬) 호출 축** → §8: 한 응답에서 독립된 도구 여러 개를 동시에 부르는지 측정합니다.

> **실행 환경**: `notebooks/.env` 에 `NVIDIA_API_KEY`, `OPENROUTER_API_KEY` 필요(로컬 서버 불필요).
> 무료 모델 결과는 **시간대·부하에 따라 달라집니다.** 아래 표의 수치는 실행할 때마다 직접 확인하세요.
> 상세는 `env_guides/M02_2_function_calling.md` 참고.

## 0. 환경 설정

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(''))
import utils
utils.reload_env()

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text          # 공급자 무관 응답 정규화(<think> 제거)

# NVIDIA build(ChatNVIDIA) + OpenRouter(ChatOpenAI, OpenAI 호환) 비교용 패키지
utils.uv_install(['langchain', 'langchain-openai', 'langchain-nvidia-ai-endpoints', 'openai'])
print('준비 완료')

## 1. 비교 모델 & 모델 팩토리

NVIDIA build 는 `ChatNVIDIA`, OpenRouter 는 OpenAI 호환이라 `ChatOpenAI(base_url=...)` 로 접속합니다.
OpenRouter 안에서 모델을 바꾸는 것은 `model` 문자열 한 줄만 바꾸는 일입니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# 비교 대상: (공급자, 모델) — 모두 무료로 사용 가능
MODELS = {
    "nvidia":  {"label": "NVIDIA deepseek-v4.1-flash",  "provider": "nvidia",     "model": utils.NVIDIA_MODEL},
    "small":   {"label": "OR lfm-2.5-2.6b (소형)",       "provider": "openrouter", "model": "liquid/lfm-2.5-2.6b:free"},
    "large":   {"label": "OR nemotron-3-super (대형)",   "provider": "openrouter", "model": "nvidia/nemotron-3-super-120b-a12b:free"},
    "router":  {"label": "OR openrouter/free (라우터)",  "provider": "openrouter", "model": "openrouter/free"},
}

def make_chat(provider, model, temperature=0.7):
    """공급자·모델에 맞는 LangChain 채팅 모델을 만든다(도구 바인딩 전).

    timeout: 무료 모델은 대기열 때문에 응답이 늦을 수 있어 넉넉히 2분으로 둔다.
    max_retries: OpenRouter 무료 모델은 분당 요청 한도(429)에 걸리기 쉬워, 자동 재시도(지수 백오프)를 늘린다.
    """
    if provider == "nvidia":
        # 사고 모드는 utils.get_llm('nvidia') 와 같이 기본으로 끈다(NVIDIA_THINKING=false)
        extra = {} if utils.NVIDIA_THINKING else {"model_kwargs": {"chat_template_kwargs": {"thinking": False}}}
        return ChatNVIDIA(model=model, api_key=utils.NVIDIA_API_KEY,
                          temperature=temperature, max_completion_tokens=1024, **extra)
    return ChatOpenAI(model=model, base_url=utils.OPENROUTER_BASE_URL, api_key=utils.OPENROUTER_API_KEY,
                      temperature=temperature, max_tokens=1024, timeout=120, max_retries=5)

def make_llm(key, temperature=0.7):
    """MODELS 의 key 로 도구가 바인딩된 LLM 을 만든다.
    (TOOLS 는 다음 셀에서 정의되며, 호출 시점에 참조된다.)
    """
    m = MODELS[key]
    return make_chat(m["provider"], m["model"], temperature).bind_tools(TOOLS)

for k, m in MODELS.items():
    print(f"  {k:7s} {m['provider']:11s} {m['model']}")

## 2. 도구 정의 + 분류기(네이티브 `tool_calls` 우선, `<tool_call>` 파싱 폴백)

도구 호출이 **반드시 필요한** 요청을 준비하고, 응답을 다음 4가지로 분류합니다.

| 분류 | 의미 |
|------|------|
| `TOOL_OK` | **실행 가능한 도구 호출** 획득 — 네이티브 `tool_calls` 또는 파싱된 `<tool_call>` JSON ✅ |
| `MALFORMED` | 도구 호출 텍스트는 있으나 JSON 이 **깨져** 파싱 불가 ⚠️ |
| `NO_TOOL` | 도구를 시도하지 않고 그냥 말로 답함 |
| `ERROR` | 예외(과부하 503, 요청 한도 429, 타임아웃 등) |

`extract_calls()` 가 응답에서 실행 가능한 호출을 뽑아냅니다. 클라우드 API 는 보통 **네이티브 `tool_calls`
(source=native)** 로 응답하지만, 일부 모델이 도구 호출을 본문 텍스트에 흘리면 `<tool_call>` 정규식
파싱(source=parsed)으로 폴백합니다. 반복 변동을 보기 위해 `temperature=0.7` 을 사용합니다.

In [ ]:
import math, time, json as _json, re
from collections import Counter
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    return f"{city} 날씨: 맑음, 22도"

# eval 에 안전한 수학 심볼만 노출 (모델이 sqrt/pow 등을 그대로 쓰는 경우 대비)
_MATH = {k: getattr(math, k) for k in
         ['sqrt','pow','sin','cos','tan','log','log10','exp','pi','e','factorial','floor','ceil','fabs']}

@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. (예: 'sqrt(144)', '12 * 12')"""
    try:
        return f"{expression} = {eval(expression, {'__builtins__': {}}, _MATH)}"
    except Exception as e:
        return f"오류: {e}"

TOOLS = [get_weather, calculator]
TOOL_FUNCS = {t.name: t for t in TOOLS}
# 이 요청은 '날씨 조회' + '제곱근 계산' 두 도구를 모두 필요로 한다(완성도 측정용).
REQUIRED_TOOLS = {"get_weather", "calculator"}
PROMPT = [SystemMessage(content="반드시 제공된 도구를 사용해 답하세요."),
          HumanMessage(content="서울 날씨를 알려주고 144의 제곱근을 계산해줘.")]

_TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def parse_text_calls(text):
    """본문 텍스트에서 <tool_call> JSON 블록을 추출·검증해 (name, args) 리스트로 반환."""
    calls = []
    for m in _TOOL_RE.finditer(text):
        try:
            obj = _json.loads(m.group(1))
        except Exception:
            continue                                  # JSON 깨짐 → 실행 불가
        if isinstance(obj, dict) and "name" in obj:
            calls.append((obj["name"], obj.get("arguments", obj.get("parameters", {})) or {}))
    return calls

def extract_calls(resp):
    """실행 가능한 도구 호출을 뽑는다. 반환: (calls, source) — source: native|parsed|None"""
    native = getattr(resp, "tool_calls", None) or []
    if native:
        return [(t["name"], t.get("args", {})) for t in native], "native"
    parsed = parse_text_calls(to_text(resp.content))  # 본문에 흘린 도구 호출 폴백
    if parsed:
        return parsed, "parsed"
    return [], None

def classify(resp):
    """응답을 TOOL_OK / MALFORMED / NO_TOOL 로 분류."""
    calls, source = extract_calls(resp)
    if calls:
        return "TOOL_OK", source, [n for n, _ in calls]
    text = to_text(resp.content)
    if ("<tool_call>" in text) or ('"arguments"' in text) or ('"parameters"' in text):
        return "MALFORMED", None, None               # 호출 의도는 있으나 파싱 실패
    return "NO_TOOL", None, None

def run_experiment(llm_with_tools, n=10, pause=0.0):
    """같은 요청을 n 번 반복하고 분류 결과(및 호출된 도구 이름·응답 모델)를 반환한다.

    pause: 호출 사이 대기 시간(초). OpenRouter 무료 모델의 분당 요청 한도(429)를 피하는 데 쓴다.
    """
    results = []
    for i in range(n):
        if i and pause:
            time.sleep(pause)
        try:
            resp = llm_with_tools.invoke(PROMPT)
            kind, source, names = classify(resp)
            served = resp.response_metadata.get("model_name", "")   # 실제로 응답한 모델(라우터 확인용)
            detail = f"{names} ({source})" if names else to_text(resp.content)[:60].replace(chr(10), ' ')
            detail += f"  · {served}" if served else ""
        except Exception as e:
            kind, source, names, detail = "ERROR", None, None, repr(e)[:80]
        results.append({"kind": kind, "names": names or [], "detail": detail})
        print(f"  run {i+1:2d}: {kind:10s} | {detail}")
    return results

def summarize(results, label):
    """유효 tool_calls 비율 + '두 도구 모두 호출'(완성도) 비율을 집계."""
    c = Counter(r["kind"] for r in results)
    n = len(results)
    ok = c.get("TOOL_OK", 0)
    both = sum(1 for r in results if REQUIRED_TOOLS.issubset(set(r["names"])))  # 완성도 지표
    print(f"\n[{label}] 유효 tool_calls {ok}/{n} = {ok/n*100:.0f}%  ·  "
          f"두 도구 모두 호출 {both}/{n} = {both/n*100:.0f}%  ·  분류 {dict(c)}")
    return {"label": label, "n": n, "ok": ok, "both": both, "counts": dict(c)}

print("도구/도우미 준비 완료:", [t.name for t in TOOLS])

## 3. 실험 ① — NVIDIA build · `deepseek-v4.1-flash` (기본) ★

본 강의의 기본 모델입니다. 모델이 고정되어 있어 결과 재현성이 높고, 두 도구를 한 응답에서
**병렬로** 호출하는 경우가 많습니다.

In [ ]:
N = 10
SUMMARY = {}   # key → summarize() 결과
OR_PAUSE = 3.0 # OpenRouter 호출 간격(초): 무료 모델 분당 한도(약 20회/분)를 넘지 않도록

def run_model(key, n=N):
    """MODELS[key] 로 반복 실험을 돌리고 요약을 SUMMARY 에 저장한다."""
    m = MODELS[key]
    print(f"=== {m['label']} 반복 실험 (N={n}) — {m['provider']} · {m['model']} ===")
    pause = OR_PAUSE if m["provider"] == "openrouter" else 0.0
    res = run_experiment(make_llm(key), n, pause)
    SUMMARY[key] = summarize(res, m["label"])
    return res

res_nvidia = run_model("nvidia")

## 4. 실험 ② — OpenRouter · 소형 `lfm-2.5-2.6b` (모델 크기 축)

2.6B 파라미터의 **소형 모델**입니다. 모델이 작아도 도구 호출을 잘 학습했다면 규격에 맞는 `tool_calls` 를
만들 수 있습니다. 무료 고정 모델이라 요청이 몰리는 시간대에는 `ERROR`(429)가 섞일 수 있습니다.

In [ ]:
res_small = run_model("small")

## 5. 실험 ③ — OpenRouter · 대형 `nemotron-3-super-120b` (모델 크기 축)

120B 파라미터의 **대형 모델**입니다. 유효한 도구 호출은 만들지만, 두 도구가 필요한 요청에서 **한 응답에
도구를 하나만** 부르는 경향이 있는지 **"두 도구 모두 호출(완성도)"** 비율로 확인합니다.
하나만 부르는 모델도 에이전트 루프를 여러 번 돌면 결국 두 작업을 처리하지만, 왕복 횟수가 늘어납니다.

In [ ]:
res_large = run_model("large")

incomplete = [x for x in res_large if x["kind"] == "TOOL_OK" and not REQUIRED_TOOLS.issubset(set(x["names"]))]
if incomplete:
    print(f"\n--- 두 도구 중 일부만 호출한(불완전) 응답 {len(incomplete)}/{N} 건 예시 ---")
    for r in incomplete[:3]:
        print("  •", r["detail"][:120])

## 6. 실험 ④ — OpenRouter · `openrouter/free` 라우터 (공급자 가용성 축)

**Free Models Router** 는 요청마다 사용 가능한 무료 모델을 고릅니다. 각 run 끝의 `· 모델명` 을 보면
호출마다 응답 모델이 바뀌는 것을 확인할 수 있습니다. 특정 모델이 과부하여도 다른 모델로 넘어가므로
`ERROR` 가 적지만, 모델이 바뀌는 만큼 **결과 재현성은 낮습니다.** 라우터도 계정 단위의 분당 요청 한도(429)는
피할 수 없으므로, 이 노트북은 OpenRouter 호출 사이에 `OR_PAUSE`(3초) 간격을 둡니다.

In [ ]:
res_router = run_model("router")

served = Counter(r["detail"].split("· ")[-1] for r in res_router if "· " in r["detail"])
print("\n응답 모델 분포:", dict(served))

### 6-1. 실제 도구 실행 루프 (NVIDIA `deepseek-v4.1-flash`)

추출한 호출이 **진짜 실행 가능**함을 보입니다: 응답 파싱 → 파이썬 도구 실행 → 결과를 모델에 되먹여 최종 답변 생성.

In [ ]:
llm_nvidia = make_llm("nvidia")
resp = llm_nvidia.invoke(PROMPT)
calls, source = extract_calls(resp)
print(f"1) 추출된 도구 호출 (source={source}):")
for name, args in calls:
    print(f"   - {name}({args})")

print("\n2) 도구 실제 실행:")
tool_results = []
for name, args in calls:
    out = TOOL_FUNCS[name].invoke(args)
    tool_results.append(f"{name} -> {out}")
    print(f"   - {out}")

print("\n3) 결과를 모델에 되먹여 최종 답변 생성:")
followup = PROMPT + [
    HumanMessage(content="도구 실행 결과입니다:\n" + "\n".join(tool_results) +
                 "\n이 결과만으로 사용자에게 한국어로 자연스럽게 최종 답변하세요.")]
# 최종 답변은 도구 없이 텍스트만 생성(기본 nvidia/deepseek-v4.1-flash)
final = utils.get_llm("nvidia", temperature=0.3).invoke(followup)
print("   ", to_text(final.content).strip()[:300])

## 7. 비교 결과 (4자)

In [ ]:
def bar(pct, width=20):
    fill = int(round(pct / 100 * width))
    return "█" * fill + "·" * (width - fill)

print(f"{'공급자 · 모델':<30}{'유효 tool_calls':<26}{'두 도구 모두(완성도)':<26}{'ERROR'}")
print("-" * 92)
for key in ["nvidia", "small", "large", "router"]:
    s = SUMMARY.get(key)
    if not s or s.get("n", 0) == 0:
        continue
    ok_pct = s["ok"] / s["n"] * 100
    both_pct = s["both"] / s["n"] * 100
    err = s["counts"].get("ERROR", 0)
    print(f"{s['label']:<30}{bar(ok_pct)} {ok_pct:3.0f}%   {bar(both_pct)} {both_pct:3.0f}%   {err}/{s['n']}")

print("\n해석 — function calling 신뢰도의 축:")
print("  (모델 크기 축)  소형 lfm-2.5-2.6b  vs  대형 nemotron-3-super  ← 같은 공급자(OpenRouter), 다른 모델")
print("     · '크면 무조건 낫다'가 아니다. 도구 호출 학습 방식에 따라 완성도(두 도구 모두 호출)가 갈린다")
print("  (공급자 가용성 축) 고정 :free 모델  vs  openrouter/free 라우터")
print("     · 고정 무료 모델은 과부하(503)·한도(429)로 ERROR 가 섞일 수 있고, 라우터는 가용성이 높은 대신 모델이 바뀐다")
print("  ⇒ 에이전트에 쓸 모델은 이름·크기만 보지 말고, 같은 요청을 N회 반복해 직접 측정하라.")

## 8. 멀티(병렬) Function Calling — 에이전트를 위한 핵심 능력

앞 실험(§3~7)은 *"유효한 도구 호출을 만드느냐"*(신뢰도)를 봤습니다. 여기서는 한 걸음 더 나아가
**"한 번의 응답에서 여러 도구를 동시에 호출할 수 있느냐"** 를 봅니다.

- **단일(single) function calling**: 한 응답에 도구를 **하나만** 요청 → 나머지는 다음 턴에.
- **멀티(병렬, parallel) function calling**: 서로 **독립적인** 여러 도구를 **한 응답에서 동시에** 요청.

### 왜 에이전트에 중요한가
독립된 하위 작업(예: "날씨 조회" + "제곱근 계산")은 서로 순서 의존이 없습니다. 멀티 호출은 이를 **한 턴에**
모아 요청하므로:
- **왕복(round-trip)·지연 감소** — LLM 호출 횟수가 줄어 더 빠르고 저렴하다.
- **동시 계획** — 모델이 필요한 행동들을 한꺼번에 세워 도구 오케스트레이션이 단순해진다.
- **ReAct/도구 에이전트 효율** — 여러 관찰을 한 번에 모아 다음 추론으로 넘어간다.

단일만 되는 모델도 에이전트 루프를 여러 번 돌면 결국 두 작업을 처리하지만, 턴 수가 늘어 느리고 실패
지점이 많습니다. 그래서 **병렬 호출 능력은 실용적인 에이전트 설계에서 중요한 요소**입니다.

> 📌 멀티/단일 여부는 *모델·프롬프트·온도* 가 함께 좌우하며 **비결정적 편차**가 있습니다. 따라서
> 특정 모델을 에이전트에 쓰기 전, **아래처럼 직접 N회 측정**해 병렬 호출이 안정적인지 확인하는 습관이 중요합니다.

In [ ]:
# === §8. 멀티(병렬) function calling 측정 — 한 응답에서 도구를 몇 개 호출하는가 ===
MULTI_USER = "서울 날씨를 알려주고 144의 제곱근을 계산해줘."
P_MULTI = [SystemMessage(content="반드시 제공된 도구를 사용하세요."), HumanMessage(content=MULTI_USER)]

def count_parallel(llm_with_tools, prompt, n=8, pause=0.0):
    """같은 요청을 n회 던져 '단일 응답에서 반환된 tool_calls 개수'를 기록한다."""
    counts = []
    for i in range(n):
        if i and pause:
            time.sleep(pause)   # OpenRouter 분당 한도(429) 회피
        try:
            resp = llm_with_tools.invoke(prompt)
            counts.append(len(getattr(resp, "tool_calls", None) or []))
        except Exception as e:
            counts.append(f"ERR:{type(e).__name__}")
    return counts

def verdict(counts):
    """응답별 호출 개수로 판정: 하나라도 2+면 병렬 가능, 전부 1이면 단일, 편차는 별도 표시."""
    nums = [c for c in counts if isinstance(c, int)]
    if not nums:
        return "측정 불가(전부 오류) ❓"
    if all(c == 0 for c in nums):
        return "도구 미호출 ❓"
    multi = sum(1 for c in nums if c >= 2)
    if multi == 0:
        return "단일만 ⚠️"
    if multi == len(nums):
        return "멀티(병렬) ✅"
    if multi * 2 >= len(nums):
        return f"주로 멀티(편차 있음: {multi}/{len(nums)}) ✅"
    return f"일부 멀티(주로 단일: 병렬 {multi}/{len(nums)}) ⚠️"

Nm = 8
print(f"요청: {MULTI_USER}\n(독립된 두 도구 get_weather·calculator 가 모두 필요, 각 {Nm}회 반복, temperature=0)\n")
PARALLEL = {}
for key, m in MODELS.items():
    counts = count_parallel(make_llm(key, temperature=0), P_MULTI, Nm,
                            OR_PAUSE if m["provider"] == "openrouter" else 0.0)
    PARALLEL[key] = verdict(counts)
    print(f"[{m['label']:30s}] 응답별 tool_calls {counts} → {PARALLEL[key]}")

print("\n해석:")
print("  · 같은 요청이라도 모델에 따라 한 응답에서 두 도구를 병렬로 부르거나(멀티), 하나씩 부른다(단일)")
print("  · 호출 개수는 모델·프롬프트·온도에 따라 런마다 달라질 수 있다(비결정적 편차)")
print("  ⇒ '특정 모델은 된다/안 된다'고 단정하지 말고 쓸 모델로 직접 측정하라. 병렬 호출은 왕복·지연을 줄여 에이전트에 중요.")

## 9. 정리

function calling 신뢰도는 **여러 축**에 함께 달려 있습니다.

1. **모델 크기 축** — 실험 ② vs ③ (같은 OpenRouter, 다른 모델):
   - 모델이 크다고 무조건 완성도가 높은 것은 아닙니다. 도구 호출을 어떻게 학습했는지에 따라
     소형 모델이 두 도구를 모두 부르고, 대형 모델이 하나만 부르는 경우도 생깁니다.
2. **공급자 가용성 축** — 실험 ②·③ vs ④:
   - 특정 `:free` 모델을 고정하면 과부하(503)·요청 한도(429)로 **`ERROR`** 가 섞일 수 있습니다.
   - `openrouter/free` 라우터는 사용 가능한 모델로 넘어가 **가용성**이 높지만, 호출마다 모델이 바뀌어 **재현성**은 낮습니다.
   - NVIDIA build 처럼 **모델을 고정**하면 재현성이 높아, 강의 기본 모델로 적합합니다.
3. **멀티(병렬) 호출 축** — §8 (한 응답에서 도구를 몇 개 호출하나):
   - 병렬 호출은 왕복·지연을 줄이고 동시 계획을 가능하게 해 **실용적 에이전트 설계에서 중요**합니다.
   - 호출 개수는 **비결정적 편차**가 있으므로, 쓸 모델로 **직접 반복 측정**하는 습관이 좋습니다.

**결론**: 에이전트에 쓸 모델은 **이름·크기가 아니라 측정으로** 고릅니다. 본 강의는 도구 호출이 안정적이고
병렬 호출을 지원하며 결과 재현성이 높은 **NVIDIA build `deepseek-ai/deepseek-v4.1-flash`** 를 기본으로 쓰고,
메인 노트북(`M02_3_mcp_a2a.ipynb` §7)은 이 기본 공급자를 `utils.get_llm()` 으로 그대로 에이전트에 사용합니다.